# Semantic Search to Moment Search

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muthukumar-Selvarasu/agentic-semantic-search/blob/main/live_test.ipynb)

Live test of both retrieval pipelines on the Steve Jobs Stanford 2005 transcript.

Open this notebook from the Colab badge above, then use **Runtime → Run all**. The first code cell prepares Colab. The next cell builds both indexes. The benchmark cell answers the two assignment questions. The last cell is the one to edit: change `LIVE_QUERY` and run it again. The indexes stay loaded.

If the YouTube caption request fails, the same cells continue on an embedded lecture and ask questions written for that lecture.


In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Muthukumar-Selvarasu/agentic-semantic-search.git"
REPO_DIR = "/content/agentic-semantic-search"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(os.path.join(REPO_DIR, "app.py")):
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("Colab is ready. The repository is installed. Run the remaining cells.")
else:
    print("Local runtime. app.py is already in this folder.")


Local runtime. app.py is already in this folder.


In [2]:
from app import Question, build_index, compare_question, format_range, print_inventory, print_scorecard, sentence_edges

index = build_index()

/Users/muthukumars/Documents/workspace/agentic-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semantic Search to Moment Search
Baseline: 250-character chunks, 50-character overlap, top-3
Moment RAG: pause and topic-cue boundaries, full moment context
Fetching YouTube transcript for Steve Jobs, Stanford commencement address (2005)
https://www.youtube.com/watch?v=UF8uR6Z6KLc


Downloaded 244 captions.

Source: YouTube UF8uR6Z6KLc
Captions: 244
Baseline chunks: 61 (61 clipped at a sentence edge) in collection baseline_collection
Moments: 15 in collection moment_collection
Moments:
  M00  00:07–00:14        opening                       This program is brought to you by Stanford University.
  M01  00:22–00:33        pause 8.0s                    Thank You.
  M02  00:36–00:46        pause 2.8s                    Truth be told I never graduated from college and this is the closest I'v
  M03  00:48–00:55        pause 2.1s                    Today I want to tell you three stories from my life.
  M04  00:56–02:53        topic cue                     The first story is about connecting the dots.
  M05  02:54–04:44        max duration                  The minute I dropped out I could stop taking the required classes that d
  M06  04:47–05:33        pause 3.6s                    If I had never dropped out, I would have never dropped in on this callig
  M07  05:39–07:3

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6655.98it/s]

Stored 61 baseline chunks and 15 moments.


## Part 1 — baseline chunks

Fixed windows are 250 characters with 50 characters of overlap. A cut can land mid-word. The first three windows below show the timestamp and whether the window starts and ends on a sentence edge.

In [3]:
print(f"Source: {index.source}")
print(f"Captions: {len(index.entries)}")
print(f"Baseline chunks: {len(index.chunks)}")
print()
for passage in index.chunks[:3]:
    starts_ok, ends_ok = sentence_edges(passage.text)
    print(format_range(passage.start_time, passage.end_time))
    print(passage.text.replace("\n", " "))
    print(f"starts on a sentence: {starts_ok}    ends on a sentence: {ends_ok}")
    print()

Source: YouTube UF8uR6Z6KLc
Captions: 244
Baseline chunks: 61

00:07–00:46
This program is brought to you by Stanford University. Please visit us at stanford.edu Thank You. I am honored to be with you today at your commencement from one of the finest universities in the world. Truth be told I never graduated from college an
starts on a sentence: True    ends on a sentence: False

00:30–01:00
d. Truth be told I never graduated from college and this is the closest I've ever gotten to a college graduation. Today I want to tell you three stories from my life. That's it. No big deal. Just three stories. The first story is about connecting the
starts on a sentence: False    ends on a sentence: False

00:52–01:14
e stories. The first story is about connecting the dots. I dropped out of Reed College after the first 6 months, but then stayed around as a drop-in for another 18 months or so before I really quit. So why did I drop out? It started before I was born
starts on a sentence: False    en

## Part 2 — moments

A moment starts on a pause of at least 2 seconds or on a topic cue. Each row is one retrievable span: index, time range, why the boundary opened, and the moment summary.

In [4]:
print_inventory(index.moments)

Moments:
  M00  00:07–00:14        opening                       This program is brought to you by Stanford University.
  M01  00:22–00:33        pause 8.0s                    Thank You.
  M02  00:36–00:46        pause 2.8s                    Truth be told I never graduated from college and this is the closest I'v
  M03  00:48–00:55        pause 2.1s                    Today I want to tell you three stories from my life.
  M04  00:56–02:53        topic cue                     The first story is about connecting the dots.
  M05  02:54–04:44        max duration                  The minute I dropped out I could stop taking the required classes that d
  M06  04:47–05:33        pause 3.6s                    If I had never dropped out, I would have never dropped in on this callig
  M07  05:39–07:36        pause 5.3s + topic cue        My second story is about love and loss.
  M08  07:36–08:56        max duration                  and fell in love with an amazing woman who would become my wife

## Benchmark comparison

The same two questions go through both indexes. Baseline answers stitch the top 3 windows and mark gaps with `[...]`. Moment RAG answers with the full top moment only.

In [5]:
score_rows = [
    compare_question(index, question, number)
    for number, question in enumerate(index.questions, start=1)
]
print()
print_scorecard(score_rows)


QUESTION 1
What did Steve Jobs learn about connecting the dots from dropping out and the calligraphy class?
BASELINE SEMANTIC RAG                          | MOMENT RAG
---------------------------------------------- | ----------------------------------------------
Retrieved timestamps:                          | Retrieved timestamps:
1. 04:51–05:07 (chunk 19)                      | 1. 04:47–05:33 (moment 6, full context)
2. 03:26–03:49 (chunk 13)                      | Also retrieved: 02:54–04:44; 13:10–14:35
3. 13:18–13:34 (chunk 55)                      | 
                                               | Context quality:
Context quality:                               | Intact single span. Key details 3/3. About 185
Non-contiguous windows. Clipped — segment 1    | tokens.
starts mid-sentence; segment 2 starts          | 
mid-sentence and ends mid-sentence; segment 3  | Answer:
starts mid-sentence and ends mid-sentence. Key | Moment summary: If I had never dropped out, I
details 3/3. A

## Live query

Edit the question in the next cell and run that cell again. This does not rebuild the indexes.

In [6]:
LIVE_QUERY = "What did Steve Jobs say about the calligraphy class?"

compare_question(index, Question(text=LIVE_QUERY, probes=[]))


LIVE QUERY
What did Steve Jobs say about the calligraphy class?
BASELINE SEMANTIC RAG                          | MOMENT RAG
---------------------------------------------- | ----------------------------------------------
Retrieved timestamps:                          | Retrieved timestamps:
1. 04:51–05:07 (chunk 19)                      | 1. 02:54–04:44 (moment 5, full context)
2. 03:26–03:49 (chunk 13)                      | Also retrieved: 04:47–05:33; 13:10–14:35
3. 03:38–03:58 (chunk 14)                      | 
                                               | Context quality:
Context quality:                               | Intact single span. Free-form query, no
Non-contiguous windows. Clipped — segment 1    | key-detail checklist. About 417 tokens.
starts mid-sentence; segment 2 starts          | 
mid-sentence and ends mid-sentence; segment 3  | Answer:
starts mid-sentence and ends mid-sentence.     | Moment summary: The minute I dropped out I
Free-form query, no key-detail check

('Live', 188, 417, 0, 0, 0)